![DB Academy](https://files.training.databricks.com/binder/prod_main/enable-trusted-insights-for-your-organization-with-genie-en_us-1.0.0/images/20261001T150649Z/Enable Trusted Insights for your Organization with Genie/Includes/images/db-academy.png)

# Demo - Create and Explore Your First Genie Agent
#### Duration - 15 Minutes

## Overview

This is your first hands-on pass with a Genie Agent, built on the **bakehouse** dataset (a fictional bakery franchise business). You will:

1. **Explore the data** the agent will use, including one data quality issue.
2. **Create a Genie Agent** on the bakehouse tables.
3. **Explore its UI** and ask a question.
4. **Review the response** and give feedback.

You start with an unconfigured agent on purpose. Benchmarking and curation come in the next notebook.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Identify the bakehouse tables** and their relationships, and spot a data quality issue.
2. **Create a Genie Agent** by selecting Unity Catalog tables and a SQL warehouse.
3. **Navigate the Genie Agent UI**, including the chat interface, configuration panel, and sources.
4. **Ask a question, interpret the generated SQL, and give feedback** with thumbs up/down and request review.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless SQL Warehouse (2X-Small is sufficient)**
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

1. Run the cell below to initialize your environment. This setup step will:
    - Assume that you have **permission to create a catalog**
    - Create a catalog named **labuser_YOUR_USER_NAME**
    - Create a schema named **genie_course_bakehouse**
    - Create the necessary tables (copies the **samples.bakehouse** data to your catalog with minor modifications)

In [0]:
%run ./Includes/Classroom-Setup-creating-genie-agent

2. Run the cell below to view the tables in your **labuser_YOUR_USER_NAME.genie_course_bakehouse** schema.

    Confirm the following 5 tables exist:

    - **media_customer_reviews_gold**
    - **sales_customers_gold**
    - **sales_franchises_gold**
    - **sales_suppliers_gold**
    - **sales_transactions_gold**

In [0]:
SHOW TABLES IN genie_course_bakehouse;


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

For this course, we assume a data pipeline is already running and producing the **gold tables** required for the Genie Agent.

These tables represent curated, analytics-ready data that Genie will use to answer questions.

  </div>
</div>

## B. Understanding the Bakehouse Dataset

Before creating a Genie Agent, it is important to understand the data that the agent will use.

> The **bakehouse** dataset represents a fictional bakery franchise business with locations around the world. It includes data on customers, franchise locations, suppliers, sales transactions, and customer reviews.

| Table | Description | Rows |
|-------|-------------|------|
| **sales_customers_gold** | Customer profiles including name, email, location, and demographics | 300 |
| **sales_franchises_gold** | Bakery franchise locations with city, country, size, and geographic coordinates | 48 |
| **sales_suppliers_gold** | Ingredient suppliers with location and approval status | 27 |
| **sales_transactions_gold** | Sales transactions including product, quantity, price, and payment method | 3,333 |
| **media_customer_reviews_gold** | Customer reviews with free-text feedback and a sentiment label | 204 |

### B1. Preview the Tables
Run the queries below to quickly preview each table and understand the data you will be working with.

1. View the **sales_customers_gold** table.

    Notice it contains:
      - a **customerID** column
      - general customer information such as name, contact details, and location

In [0]:
-- Preview the sales_customers table
SELECT *
FROM sales_customers_gold
LIMIT 10;

2. View the **sales_franchises_gold** table.

    Notice it contains:
      - a **franchiseID** column
      - location details such as city, district, zip code, and country
      - franchise attributes such as size and supplier relationships (**supplierID** column)

In [0]:
-- Preview the sales_franchises table
SELECT *
FROM sales_franchises_gold;

<div style="max-width: 1150px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- HEADER BAND -->
<div style="background: #1B5162; color: white; border-radius: 10px 10px 4px 4px; padding: 20px 26px; margin-bottom: 16px;">
  <div style="font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #B7CDD5; margin-bottom: 6px;">Context Your Agent Will Need</div>
  <div style="font-size: 21pt; font-weight: 700; line-height: 1.25;">Two things to teach the Genie Agent</div>
  <div style="font-size: 15pt; opacity: 0.92; margin-top: 6px; line-height: 1.45;">Previewing <strong>sales_franchises_gold</strong> surfaces two details the agent cannot infer on its own. Each becomes context you add during curation.</div>
</div>

<!-- TWO ISSUE COLUMNS -->
<div style="display: flex; gap: 16px; flex-wrap: nowrap; margin-bottom: 16px;">

  <!-- ISSUE 1: DUPLICATE NAMES -->
  <div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 20px 22px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06); box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #FFAB00; border-radius: 8px 8px 0 0;"></div>
    <div style="display: flex; align-items: center; gap: 10px; margin: 8px 0 12px 0;">
      <span style="flex-shrink: 0; width: 30px; height: 30px; border-radius: 50%; background: #FFAB00; color: white; font-weight: 800; font-size: 15pt; display: flex; align-items: center; justify-content: center;">1</span>
      <span style="font-size: 17pt; font-weight: 700;">Names are not unique</span>
    </div>
    <div style="font-size: 14pt; line-height: 1.55;">
      The same <code>name</code> can belong to two different franchises, each with its own <code>franchiseID</code> and city.
    </div>
    <table style="width: 100%; border-collapse: separate; border-spacing: 0; margin-top: 12px; border-radius: 6px; overflow: hidden; font-size: 13pt;">
      <thead>
        <tr style="background: #1B5162; color: white;">
          <th style="padding: 7px 10px; text-align: left;">name</th>
          <th style="padding: 7px 10px; text-align: left;">franchiseID</th>
          <th style="padding: 7px 10px; text-align: left;">city</th>
        </tr>
      </thead>
      <tbody>
        <tr style="background: rgba(255,171,0,0.14);"><td style="padding: 7px 10px;">Sugar Rush</td><td style="padding: 7px 10px;"><code>3000003</code></td><td style="padding: 7px 10px;">Melbourne</td></tr>
        <tr style="background: rgba(255,171,0,0.14);"><td style="padding: 7px 10px;">Sugar Rush</td><td style="padding: 7px 10px;"><code>3000044</code></td><td style="padding: 7px 10px;">Berlin</td></tr>
        <tr style="background: rgba(66,153,224,0.12);"><td style="padding: 7px 10px;">Sweet Temptations</td><td style="padding: 7px 10px;"><code>3000014</code></td><td style="padding: 7px 10px;">Seattle</td></tr>
        <tr style="background: rgba(66,153,224,0.12);"><td style="padding: 7px 10px;">Sweet Temptations</td><td style="padding: 7px 10px;"><code>3000032</code></td><td style="padding: 7px 10px;">New Orleans</td></tr>
      </tbody>
    </table>
    <div style="margin-top: 12px; background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; border-radius: 6px; padding: 10px 12px; font-size: 14pt; line-height: 1.5;">
      <strong style="color: #087F55;">Give Genie:</strong> group by <code>franchiseID</code> and city, never by name alone.
    </div>
  </div>

  <!-- ISSUE 2: MISSING REGION -->
  <div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 20px 22px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06); box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #4299E0; border-radius: 8px 8px 0 0;"></div>
    <div style="display: flex; align-items: center; gap: 10px; margin: 8px 0 12px 0;">
      <span style="flex-shrink: 0; width: 30px; height: 30px; border-radius: 50%; background: #4299E0; color: white; font-weight: 800; font-size: 15pt; display: flex; align-items: center; justify-content: center;">2</span>
      <span style="font-size: 17pt; font-weight: 700;">Regions are not in the data</span>
    </div>
    <div style="font-size: 14pt; line-height: 1.55;">
      There is a <code>country</code> column but no <code>region</code>. The business groups countries into three regions:
    </div>
    <div style="margin-top: 12px;">
      <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 8px;">
        <span style="flex-shrink: 0; width: 68px; text-align: center; background: #4299E0; color: white; font-weight: 700; font-size: 13pt; padding: 5px 6px; border-radius: 6px;">APJ</span>
        <span style="font-size: 13.5pt;">Japan, Australia</span>
      </div>
      <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 8px;">
        <span style="flex-shrink: 0; width: 68px; text-align: center; background: #00A972; color: white; font-weight: 700; font-size: 13pt; padding: 5px 6px; border-radius: 6px;">AMER</span>
        <span style="font-size: 13.5pt;">US, Canada</span>
      </div>
      <div style="display: flex; align-items: center; gap: 10px;">
        <span style="flex-shrink: 0; width: 68px; text-align: center; background: #FF5F46; color: white; font-weight: 700; font-size: 13pt; padding: 5px 6px; border-radius: 6px;">EMEA</span>
        <span style="font-size: 13.5pt;">Netherlands, France, Germany, Italy, Sweden</span>
      </div>
    </div>
    <div style="margin-top: 12px; background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; border-radius: 6px; padding: 10px 12px; font-size: 14pt; line-height: 1.5;">
      <strong style="color: #087F55;">Give Genie:</strong> a <code>region</code> SQL expression mapping country to APJ / AMER / EMEA.
    </div>
  </div>

</div>

<!-- TAKEAWAY -->
<div style="background: #FFF6F4; border-left: 4px solid #FF5F46; border-radius: 6px; padding: 14px 20px; font-size: 14pt; line-height: 1.55;">
  <span style="font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; color: #B83A24; text-transform: uppercase; display: block; margin-bottom: 4px;">Why this matters</span>
  Neither detail is visible from column names alone, and both change the answer to a real question. Exploring the data is how you find them. Later, you will curate them in: a <strong>description and instruction</strong> for the duplicate names, and a <strong>SQL expression</strong> for region.
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

### 1. Names are not unique

- `sales_franchises_gold` has 48 rows, but `name` repeats across locations:
  - **Sugar Rush** &rarr; `3000003` (Melbourne, Australia) and `3000044` (Berlin, Germany)
  - **Sweet Temptations** &rarr; `3000014` (Seattle, US) and `3000032` (New Orleans, US)
- If Genie groups by `name`, the two Sugar Rush stores collapse into one row and any per-location metric (revenue, review counts, rankings) is wrong while still looking reasonable.
- **Curate it as:** a column description on `name` (not unique) and `franchiseID` (the unique key), plus a general instruction to group by `franchiseID` and city.

### 2. Regions are not in the data

- The table records `country`, but the organization reports on **regions**, and no `region` column exists.
- The business rule is a fixed mapping: **APJ** (Japan, Australia), **AMER** (US, Canada), **EMEA** (Netherlands, France, Germany, Italy, Sweden).
- Ask "what are total sales by region?" and Genie has nothing to map to. It falls back to `country` or invents regions, and the answer can change run to run.
- **Curate it as:** a reusable `region` SQL expression in the Knowledge Store, which also becomes a benchmark question.

```
CASE
  WHEN country IN ('Japan','Australia') THEN 'APJ'
  WHEN country IN ('US','Canada') THEN 'AMER'
  WHEN country IN ('Netherlands','France','Germany','Italy','Sweden') THEN 'EMEA'
  ELSE 'Other'
END AS region
```

### The pattern

Explore the data, find what the agent cannot know on its own, then hand it that context during curation. Descriptions and instructions fix what the data says; SQL expressions supply what the data does not contain.

</details>



3. View the **sales_suppliers_gold** table.

    Notice it contains:
      - a **supplierID** column
      - supplier details such as name, ingredient, and location
      - operational attributes such as size and approval status

In [0]:
-- Preview the sales_suppliers table
SELECT *
FROM sales_suppliers_gold
LIMIT 10;

4. View the **sales_transactions_gold** table.

    Notice it contains:
      - a **transactionID** column
      - relationships to customers and franchises through **customerID** and **franchiseID**
      - transactional details such as product, quantity, pricing, and total amount
      - payment information such as payment method and card number

In [0]:
-- Preview the sales_transactions table
SELECT *
FROM sales_transactions_gold
LIMIT 10;

5. View the **media_customer_reviews_gold** table.

    Notice it contains:
      - a **review** column with unstructured text data
      - a **ref** column linking reviews to specific locations (**sales_franchises_gold.franchiseID**)
      - a **review_date** for when the review was created
      - a **flag** classification indicating positive, negative, or mixed feedback
        - Created for you using the `ai_analyze_sentiment` function:
        [AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_analyze_sentiment) |
        [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_analyze_sentiment) |
        [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_analyze_sentiment)


In [0]:
SELECT *
FROM media_customer_reviews_gold
LIMIT 10;

### B2. Check for a Data Quality Issue

Join the **sales_franchises_gold** and **sales_suppliers_gold** tables to check whether every franchise has a documented supplier.

In [0]:
SELECT
  f.franchiseID,
  COUNT(s.supplierID) AS supplier_count
FROM sales_franchises_gold f
LEFT JOIN sales_suppliers_gold s 
  ON f.supplierID = s.supplierID
GROUP BY f.franchiseID

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- HEADER BAND -->
<div style="background: #1B5162; color: white; border-radius: 10px 10px 4px 4px; padding: 20px 26px; margin-bottom: 16px;">
  <div style="font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #B7CDD5; margin-bottom: 6px;">Example: Context Your Agent Will Need</div>
  <div style="font-size: 21pt; font-weight: 700; line-height: 1.25;">Not every franchise has a matching supplier</div>
  <div style="font-size: 15pt; opacity: 0.92; margin-top: 6px; line-height: 1.45;">This is exactly the kind of data quality detail you must teach a Genie Agent, so it joins these tables correctly.</div>
</div>

<!-- TWO CARDS -->
<div style="display: flex; gap: 16px; flex-wrap: nowrap; margin-bottom: 16px;">

  <!-- CARD A: THE ISSUE -->
  <div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 20px 22px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06); box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #FFAB00; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 17pt; font-weight: 700; margin: 8px 0 12px 0;">The data quality issue</div>
    <div style="font-size: 14pt; line-height: 1.6;">
      Only <strong>27 of 48</strong> franchises have a <code>supplierID</code> that matches a real record in <strong>sales_suppliers_gold</strong>. The other <strong>21</strong> point to suppliers that do not exist.
    </div>
    <table style="width: 100%; border-collapse: separate; border-spacing: 0; margin-top: 14px; border-radius: 6px; overflow: hidden; font-size: 14pt;">
      <thead>
        <tr style="background: #1B5162; color: white;">
          <th style="padding: 8px 12px; text-align: left;">Has a valid supplier?</th>
          <th style="padding: 8px 12px; text-align: right;">Franchises</th>
        </tr>
      </thead>
      <tbody>
        <tr style="background: rgba(152,16,42,0.08);">
          <td style="padding: 8px 12px;">No</td>
          <td style="padding: 8px 12px; text-align: right; font-weight: 700;">21</td>
        </tr>
        <tr style="background: rgba(0,169,114,0.10);">
          <td style="padding: 8px 12px;">Yes</td>
          <td style="padding: 8px 12px; text-align: right; font-weight: 700;">27</td>
        </tr>
        <tr style="background: #EEEDE9;">
          <td style="padding: 8px 12px; font-weight: 700;">Total</td>
          <td style="padding: 8px 12px; text-align: right; font-weight: 700;">48</td>
        </tr>
      </tbody>
    </table>
  </div>

  <!-- CARD B: CONTEXT TO GIVE THE AGENT -->
  <div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 20px 22px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06); box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #00A972; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 17pt; font-weight: 700; margin: 8px 0 12px 0;">The context to give the Genie Agent</div>
    <div style="font-size: 14pt; line-height: 1.6;">
      Tell the agent <strong>which join to use</strong> when connecting franchises to suppliers:
    </div>
    <div style="margin-top: 12px; background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; border-radius: 6px; padding: 12px 14px; font-size: 14pt; line-height: 1.5;">
      <strong style="color: #087F55;">Use a LEFT JOIN</strong><br>
      Keeps all <strong>48</strong> franchises, including the 21 with no matching supplier.
    </div>
    <div style="margin-top: 10px; background: rgba(152,16,42,0.08); border-left: 4px solid #98102A; border-radius: 6px; padding: 12px 14px; font-size: 14pt; line-height: 1.5;">
      <strong style="color: #98102A;">Avoid an INNER JOIN</strong><br>
      It silently drops those 21 franchises, undercounting your results.
    </div>
  </div>

</div>

<!-- TAKEAWAY -->
<div style="background: #FFF6F4; border-left: 4px solid #FF5F46; border-radius: 6px; padding: 14px 20px; font-size: 14pt; line-height: 1.55;">
  <span style="font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; color: #B83A24; text-transform: uppercase; display: block; margin-bottom: 4px;">Why this matters</span>
  Exploring your data is how you find details like this. Left unaddressed, Genie may pick the wrong join and return numbers that look right but are quietly incomplete. Later, you will curate this into the agent as a <strong>join instruction</strong> and a <strong>column description</strong>.
</div>

</div>

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    A Note on Data Exploration
  </strong>
  <div style="color:#333;">

In production scenarios, it is important to **thoroughly explore and profile your data** before building a Genie Agent. 

This means understanding the distributions, nulls, edge cases, and business logic embedded in the data.

For this training, we have done a **brief, focused exploration**. Just enough to understand the tables and one key data quality issue before moving on to Genie.

  </div>
</div>


## C. Genie Agent Project Architecture
<br></br>
<div style="max-width: 1280px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- TITLE BAR -->
<div style="background: #1B5162; color: white; border-radius: 8px; padding: 20px 28px; margin-bottom: 18px;">
  <div style="font-size: 22pt; font-weight: 700; margin-bottom: 6px;">Project Architecture</div>
  <div style="font-size: 15pt; font-weight: 400; line-height: 1.45; opacity: 0.95;">Let&rsquo;s view the project architecture for our Genie Agent. The goal is to link all of our Unity Catalog tables to a Genie Agent where consumers can <b>ask questions in natural language to obtain results.</b></div>
</div>

<!-- BEST PRACTICE CALLOUT -->
<div style="background: #FFF6F4; border-left: 4px solid #FF5F46; border-radius: 6px; padding: 12px 18px; margin-bottom: 22px; font-size: 14pt; line-height: 1.5;">
  <span style="font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; color: #B83A24; text-transform: uppercase; display: block; margin-bottom: 4px;">Best Practice</span>
  It&rsquo;s best for a <b>data domain expert</b> to set up a Genie Agent with <b>stakeholder involvement</b>.
</div>

<!-- FAN-IN/FAN-OUT ARCHITECTURE -->
<div style="display: grid; grid-template-columns: 1.35fr auto 1fr auto 0.85fr; gap: 0; align-items: center;">

  <!-- LEFT COLUMN: SOURCE TABLES -->
  <div>
    <div style="font-size: 12pt; font-weight: 700; letter-spacing: 2px; color: #1B5162; text-transform: uppercase; text-align: center; margin-bottom: 12px;">Unity Catalog Tables</div>
    <div style="display: flex; flex-direction: column; gap: 10px;">
    <div style="background: #FFFFFF; border: 2px solid #1B5162; border-left: 6px solid #1B5162; border-radius: 8px; padding: 10px 14px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 4px;">
        <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#1B5162" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><ellipse cx="12" cy="5" rx="9" ry="3"></ellipse><path d="M3 5v6c0 1.66 4.03 3 9 3s9-1.34 9-3V5"></path><path d="M3 11v6c0 1.66 4.03 3 9 3s9-1.34 9-3v-6"></path></svg>
        <div style="font-family: monospace; font-size: 14pt; font-weight: 700; color: #0b2026;">sales_customers_gold</div>
      </div>
      <div style="font-size: 14pt; color: #5A6F77; font-style: italic; line-height: 1.35; padding-left: 30px;">Customer profiles and locations</div>
    </div>
    <div style="background: #FFFFFF; border: 2px solid #1B5162; border-left: 6px solid #1B5162; border-radius: 8px; padding: 10px 14px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 4px;">
        <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#1B5162" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><ellipse cx="12" cy="5" rx="9" ry="3"></ellipse><path d="M3 5v6c0 1.66 4.03 3 9 3s9-1.34 9-3V5"></path><path d="M3 11v6c0 1.66 4.03 3 9 3s9-1.34 9-3v-6"></path></svg>
        <div style="font-family: monospace; font-size: 14pt; font-weight: 700; color: #0b2026;">sales_franchises_gold</div>
      </div>
      <div style="font-size: 14pt; color: #5A6F77; font-style: italic; line-height: 1.35; padding-left: 30px;">Franchise locations and size</div>
    </div>
    <div style="background: #FFFFFF; border: 2px solid #1B5162; border-left: 6px solid #1B5162; border-radius: 8px; padding: 10px 14px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 4px;">
        <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#1B5162" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><ellipse cx="12" cy="5" rx="9" ry="3"></ellipse><path d="M3 5v6c0 1.66 4.03 3 9 3s9-1.34 9-3V5"></path><path d="M3 11v6c0 1.66 4.03 3 9 3s9-1.34 9-3v-6"></path></svg>
        <div style="font-family: monospace; font-size: 14pt; font-weight: 700; color: #0b2026;">sales_suppliers_gold</div>
      </div>
      <div style="font-size: 14pt; color: #5A6F77; font-style: italic; line-height: 1.35; padding-left: 30px;">Ingredient suppliers</div>
    </div>
    <div style="background: #FFFFFF; border: 2px solid #1B5162; border-left: 6px solid #1B5162; border-radius: 8px; padding: 10px 14px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 4px;">
        <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#1B5162" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><ellipse cx="12" cy="5" rx="9" ry="3"></ellipse><path d="M3 5v6c0 1.66 4.03 3 9 3s9-1.34 9-3V5"></path><path d="M3 11v6c0 1.66 4.03 3 9 3s9-1.34 9-3v-6"></path></svg>
        <div style="font-family: monospace; font-size: 14pt; font-weight: 700; color: #0b2026;">sales_transactions_gold</div>
      </div>
      <div style="font-size: 14pt; color: #5A6F77; font-style: italic; line-height: 1.35; padding-left: 30px;">Sales transactions and revenue</div>
    </div>
    <div style="background: #FFFFFF; border: 2px solid #1B5162; border-left: 6px solid #1B5162; border-radius: 8px; padding: 10px 14px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 4px;">
        <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#1B5162" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><ellipse cx="12" cy="5" rx="9" ry="3"></ellipse><path d="M3 5v6c0 1.66 4.03 3 9 3s9-1.34 9-3V5"></path><path d="M3 11v6c0 1.66 4.03 3 9 3s9-1.34 9-3v-6"></path></svg>
        <div style="font-family: monospace; font-size: 14pt; font-weight: 700; color: #0b2026;">media_customer_reviews_gold</div>
      </div>
      <div style="font-size: 14pt; color: #5A6F77; font-style: italic; line-height: 1.35; padding-left: 30px;">Reviews and sentiment</div>
    </div>
    </div>
  </div>

  <!-- LEFT ARROW BUNDLE -->
  <div style="display: flex; flex-direction: column; align-items: center; padding: 0 14px;">
    <div style="font-size: 32pt; color: #618794; line-height: 1;">&rarr;</div>
    <div style="font-size: 12pt; color: #618794; font-style: italic; margin-top: 4px; text-align: center; max-width: 90px;">governed data</div>
  </div>

  <!-- CENTER: GENIE HUB -->
  <div style="background: #F9F7F4; border: 3px solid #FF5F46; border-radius: 14px; padding: 22px 18px; box-shadow: 0 6px 20px rgba(255,95,70,0.18); text-align: center; position: relative;">
    <div style="position: absolute; top: -12px; left: 50%; transform: translateX(-50%); background: #FF5F46; color: white; font-size: 11pt; font-weight: 700; letter-spacing: 1.5px; padding: 3px 14px; border-radius: 12px;">GENIE AGENT</div>
    <div style="margin: 10px 0 14px 0;">
      
    </div>
    <div style="font-size: 16pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-bottom: 10px;">Ask questions in<br>natural language</div>
    <div style="background: #FFFFFF; border: 1.5px dashed #FF5F46; border-radius: 18px 18px 18px 4px; padding: 8px 14px; font-size: 13pt; font-style: italic; color: #5A6F77; text-align: left; margin-top: 12px;">
      &ldquo;What were our top 5 selling products last quarter?&rdquo;
    </div>
  </div>

  <!-- RIGHT ARROW BUNDLE -->
  <div style="display: flex; flex-direction: column; align-items: center; padding: 0 14px;">
    <div style="font-size: 32pt; color: #618794; line-height: 1;">&rarr;</div>
    <div style="font-size: 12pt; color: #618794; font-style: italic; margin-top: 4px; text-align: center; max-width: 90px;">answers &amp; charts</div>
  </div>

  <!-- RIGHT COLUMN: CONSUMERS -->
  <div>
    <div style="font-size: 12pt; font-weight: 700; letter-spacing: 2px; color: #00A972; text-transform: uppercase; text-align: center; margin-bottom: 12px;">Consumers</div>
    <div style="display: flex; flex-direction: column; gap: 12px;">
    <div style="background: #FFFFFF; border: 2px solid #00A972; border-left: 6px solid #00A972; border-radius: 8px; padding: 14px 16px; display: flex; align-items: center; gap: 12px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <svg width="26" height="26" viewBox="0 0 24 24" fill="none" stroke="#00A972" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2"></path><circle cx="12" cy="7" r="4"></circle></svg>
      <div style="font-size: 15pt; font-weight: 700; color: #0b2026;">Analysts</div>
    </div>
    <div style="background: #FFFFFF; border: 2px solid #00A972; border-left: 6px solid #00A972; border-radius: 8px; padding: 14px 16px; display: flex; align-items: center; gap: 12px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <svg width="26" height="26" viewBox="0 0 24 24" fill="none" stroke="#00A972" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2"></path><circle cx="12" cy="7" r="4"></circle></svg>
      <div style="font-size: 15pt; font-weight: 700; color: #0b2026;">Business Users</div>
    </div>
    <div style="background: #FFFFFF; border: 2px solid #00A972; border-left: 6px solid #00A972; border-radius: 8px; padding: 14px 16px; display: flex; align-items: center; gap: 12px; box-shadow: 0 1px 4px rgba(27,49,57,0.06);">
      <svg width="26" height="26" viewBox="0 0 24 24" fill="none" stroke="#00A972" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2"></path><circle cx="12" cy="7" r="4"></circle></svg>
      <div style="font-size: 15pt; font-weight: 700; color: #0b2026;">Franchise Managers</div>
    </div>
    </div>
  </div>

</div>

<!-- BOTTOM BAND: foundation note -->
<div style="margin-top: 24px; background: linear-gradient(135deg, #1B5162 0%, #0b2026 100%); color: #ffffff; border-radius: 8px; padding: 14px 22px; text-align: center;">
  <div style="font-size: 11pt; letter-spacing: 2.5px; text-transform: uppercase; color: #B7CDD5; font-weight: 700; margin-bottom: 4px;">Foundation</div>
  <div style="font-size: 14pt; color: #ffffff;">Unity Catalog &middot; one governed copy of your data &middot; permissions enforced end to end</div>
</div>

</div>

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Metric Views as a Source for Genie Agents are Recommended
  </strong>
  <div style="color:#333;">

Instead of referencing tables directly in Genie, **it's recommended that you use Metric Views**. However, to keep this course focused on **Genie**, we will reference tables directly.

**Learning metric views in depth is outside the scope of this course.**

**Metric Views** are recommended as a source for Genie Agents because they centralize business metrics, dimensions, joins, and aggregations, helping produce more accurate and consistent answers. The same governed semantic definitions can also be reused across other Databricks experiences, such as alerts and more.

<div style="width: 100%; max-width: 1200px; margin: 16px auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px 8px 4px 4px; padding: 20px 24px; text-align: center; margin-bottom: 16px;">
<div style="font-size: 20pt; font-weight: 700; margin-bottom: 8px;">What are Databricks Metric Views?</div>
<div style="font-size: 14pt; font-weight: 400; line-height: 1.5; opacity: 0.95;">A governed semantic layer that standardizes how business metrics and dimensions are defined
across your organization, built from joins, filters, and aggregations, plus metadata for AI and BI.</div>
</div>

<svg viewBox="0 110 1200 650" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block; background: #FFFFFF; border-radius: 10px;">
<defs>
<marker id="mvo-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" fill="#5E7077"/></marker>
</defs>
<rect x="20" y="120" width="780" height="400" rx="10" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
<text x="410" y="150" text-anchor="middle" font-size="17" font-weight="700" fill="#1B5162" letter-spacing="2">HOW A METRIC VIEW WORKS</text>
<text x="130" y="180" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">1. Curate your data</text>
<text x="130" y="202" text-anchor="middle" font-size="14" fill="#5E7077">Tables in Unity Catalog</text>
<rect x="50" y="218" width="160" height="80" rx="6" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="130" y="244" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Fact tables</text>
<text x="130" y="268" text-anchor="middle" font-size="14" fill="#5E7077">events, transactions,</text>
<text x="130" y="286" text-anchor="middle" font-size="14" fill="#5E7077">orders</text>
<rect x="50" y="308" width="160" height="80" rx="6" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="130" y="334" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Dimension tables</text>
<text x="130" y="358" text-anchor="middle" font-size="14" fill="#5E7077">customers, products,</text>
<text x="130" y="376" text-anchor="middle" font-size="14" fill="#5E7077">dates</text>
<rect x="50" y="398" width="160" height="80" rx="6" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="130" y="424" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Source filters</text>
<text x="130" y="448" text-anchor="middle" font-size="14" fill="#5E7077">scope and quality</text>
<text x="130" y="466" text-anchor="middle" font-size="14" fill="#5E7077">rules</text>
<line x1="216" y1="326" x2="248" y2="326" stroke="#5E7077" stroke-width="2" marker-end="url(#mvo-arr)"/>
<text x="380" y="180" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">2. Define your metric view</text>
<text x="380" y="202" text-anchor="middle" font-size="14" fill="#5E7077">Data, joins, filters, dimensions</text>
<rect x="252" y="218" width="256" height="216" rx="10" fill="none" stroke="#4299E0" stroke-width="1.5" stroke-opacity="0.35"/>
<rect x="258" y="224" width="244" height="204" rx="8" fill="#F4FAFF" stroke="#4299E0" stroke-width="2.5"/>
<text x="380" y="250" text-anchor="middle" font-size="18" font-weight="700" fill="#1B5162" letter-spacing="1">METRIC VIEW</text>
<rect x="266" y="266" width="72" height="28" rx="14" fill="#D8EBFB" stroke="#4299E0" stroke-width="1"/>
<text x="302" y="285" text-anchor="middle" font-size="14" fill="#1B5162" font-weight="700">JOINS</text>
<rect x="344" y="266" width="72" height="28" rx="14" fill="#D8EBFB" stroke="#4299E0" stroke-width="1"/>
<text x="380" y="285" text-anchor="middle" font-size="14" fill="#1B5162" font-weight="700">FILTERS</text>
<rect x="422" y="266" width="72" height="28" rx="14" fill="#D8EBFB" stroke="#4299E0" stroke-width="1"/>
<text x="458" y="285" text-anchor="middle" font-size="14" fill="#1B5162" font-weight="700">FIELDS</text>
<rect x="264" y="304" width="88" height="28" rx="14" fill="#D8EBFB" stroke="#4299E0" stroke-width="1"/>
<text x="308" y="323" text-anchor="middle" font-size="14" fill="#1B5162" font-weight="700">MEASURES</text>
<rect x="358" y="304" width="140" height="28" rx="14" fill="#D8EBFB" stroke="#4299E0" stroke-width="1"/>
<text x="428" y="323" text-anchor="middle" font-size="14" fill="#1B5162" font-weight="700">AGENTIC METADATA</text>
<text x="380" y="372" text-anchor="middle" font-size="14" fill="#5E7077">Defined once, reused</text>
<text x="380" y="392" text-anchor="middle" font-size="14" fill="#5E7077">across consumers</text>
<line x1="510" y1="326" x2="544" y2="326" stroke="#5E7077" stroke-width="2" marker-end="url(#mvo-arr)"/>
<text x="652" y="180" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">3. Use everywhere</text>
<text x="652" y="202" text-anchor="middle" font-size="14" fill="#5E7077">One source of truth</text>
<rect x="552" y="218" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#00A972" stroke-width="1.5"/>
<text x="652" y="240" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Dashboards</text>
<text x="652" y="258" text-anchor="middle" font-size="14" fill="#5E7077">AI/BI Dashboards</text>
<rect x="552" y="274" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#00A972" stroke-width="1.5"/>
<text x="652" y="296" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Genie</text>
<text x="652" y="314" text-anchor="middle" font-size="14" fill="#5E7077">Code, Agents, One</text>
<rect x="552" y="330" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#00A972" stroke-width="1.5"/>
<text x="652" y="352" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Notebooks and SQL</text>
<text x="652" y="370" text-anchor="middle" font-size="14" fill="#5E7077">ad hoc analysis</text>
<rect x="552" y="386" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#00A972" stroke-width="1.5"/>
<text x="652" y="408" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">APIs and Integrations</text>
<text x="652" y="426" text-anchor="middle" font-size="14" fill="#5E7077">downstream tools</text>
<rect x="820" y="120" width="360" height="400" rx="10" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
<text x="1000" y="150" text-anchor="middle" font-size="17" font-weight="700" fill="#1B5162" letter-spacing="2">AGENTIC METADATA</text>
<text x="1000" y="180" text-anchor="middle" font-size="14" fill="#5E7077">Metric views ship with rich metadata,</text>
<text x="1000" y="200" text-anchor="middle" font-size="14" fill="#5E7077">so agents and dashboards understand</text>
<text x="1000" y="220" text-anchor="middle" font-size="14" fill="#5E7077">your business terms correctly.</text>
<rect x="840" y="240" width="320" height="60" rx="6" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.5"/>
<rect x="840" y="240" width="6" height="60" fill="#4299E0"/>
<text x="858" y="266" font-size="16" font-weight="700" fill="#0b2026">Display names</text>
<text x="858" y="288" font-size="14" fill="#5E7077">Friendly labels for metrics and dimensions.</text>
<rect x="840" y="308" width="320" height="60" rx="6" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.5"/>
<rect x="840" y="308" width="6" height="60" fill="#4299E0"/>
<text x="858" y="334" font-size="16" font-weight="700" fill="#0b2026">Formatting</text>
<text x="858" y="356" font-size="14" fill="#5E7077">Auto-applied currency and date formats.</text>
<rect x="840" y="376" width="320" height="60" rx="6" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.5"/>
<rect x="840" y="376" width="6" height="60" fill="#4299E0"/>
<text x="858" y="402" font-size="16" font-weight="700" fill="#0b2026">Comments</text>
<text x="858" y="424" font-size="14" fill="#5E7077">Plain-language descriptions for context.</text>
<rect x="840" y="444" width="320" height="60" rx="6" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.5"/>
<rect x="840" y="444" width="6" height="60" fill="#4299E0"/>
<text x="858" y="470" font-size="16" font-weight="700" fill="#0b2026">Synonyms</text>
<text x="858" y="492" font-size="14" fill="#5E7077">Alternative terms for the same concept.</text>
<rect x="20" y="540" width="1160" height="180" rx="10" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
<text x="600" y="570" text-anchor="middle" font-size="17" font-weight="700" fill="#1B5162" letter-spacing="2">KEY BENEFITS</text>
<circle cx="160" cy="620" r="30" fill="#00A972"/>
<text x="160" y="630" text-anchor="middle" font-size="24" font-weight="700" fill="#FFFFFF">&#10003;</text>
<text x="160" y="672" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">Trusted and Consistent</text>
<text x="160" y="694" text-anchor="middle" font-size="14" fill="#5E7077">one definition, everywhere</text>
<circle cx="380" cy="620" r="30" fill="#1B5162"/>
<text x="380" y="627" text-anchor="middle" font-size="14" font-weight="700" fill="#FFFFFF">FAST</text>
<text x="380" y="672" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">Save Time</text>
<text x="380" y="694" text-anchor="middle" font-size="14" fill="#5E7077">no more rebuilding logic</text>
<circle cx="600" cy="620" r="30" fill="#4299E0"/>
<text x="600" y="627" text-anchor="middle" font-size="14" font-weight="700" fill="#FFFFFF">AI</text>
<text x="600" y="672" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">AI-Ready</text>
<text x="600" y="694" text-anchor="middle" font-size="14" fill="#5E7077">Genie picks the right field</text>
<circle cx="820" cy="620" r="30" fill="#FFAB00"/>
<text x="820" y="630" text-anchor="middle" font-size="26" font-weight="700" fill="#FFFFFF">&#9733;</text>
<text x="820" y="672" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">Better Decisions</text>
<text x="820" y="694" text-anchor="middle" font-size="14" fill="#5E7077">trustworthy metrics</text>
<circle cx="1040" cy="620" r="30" fill="#FF5F46"/>
<text x="1040" y="627" text-anchor="middle" font-size="14" font-weight="700" fill="#FFFFFF">UC</text>
<text x="1040" y="672" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">Scale with Governance</text>
<text x="1040" y="694" text-anchor="middle" font-size="14" fill="#5E7077">UC permissions and lineage</text>
<text x="600" y="744" text-anchor="middle" font-size="14" font-style="italic" fill="#5E7077">Metric views bring together governed data, business logic, and rich metadata.</text>
</svg>

<div style="background: #1B5162; color: #FFFFFF; border-radius: 4px 4px 8px 8px; padding: 14px 24px; text-align: center; font-size: 14pt; margin-top: 12px;">
<div style="margin-bottom: 10px;">
<strong style="font-weight: 700;">Unity Catalog Metric Views docs</strong>&nbsp;&nbsp;
<a href="https://docs.databricks.com/aws/en/metric-views/" target="_blank" style="color: #FFFFFF; font-weight: 700; text-decoration: underline; margin: 0 4px;">AWS</a> |
<a href="https://learn.microsoft.com/en-us/azure/databricks/metric-views/" target="_blank" style="color: #FFFFFF; font-weight: 700; text-decoration: underline; margin: 0 4px;">Azure</a> |
<a href="https://docs.databricks.com/gcp/en/metric-views/" target="_blank" style="color: #FFFFFF; font-weight: 700; text-decoration: underline; margin: 0 4px;">GCP</a>
</div>
<div style="border-top: 1px solid rgba(255,255,255,0.25); padding-top: 10px;">
Want to go deeper? Check out the <a href="https://www.databricks.com/training/catalog/building-semantic-models-in-databricks-with-uc-metric-views-4962" target="_blank" style="color: #FFFFFF; font-weight: 700; text-decoration: underline;">Building Semantic Models in Databricks with UC Metric Views</a> Databricks training.
</div>
</div>
</div>

**NOTE:** In this course, we focus on building and curating a Genie Agent directly. In production, many teams layer Genie on top of metric views to ensure consistent business semantics.

  </div>
</div>


## D. The Genie Agent Authoring Checklist
<br></br>

<div style="max-width: 1280px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- TITLE BAR -->
<div style="background: #1B5162; color: white; border-radius: 8px; padding: 20px 28px;">
  <div style="font-size: 22pt; font-weight: 700; margin-bottom: 6px;">Genie Agent Authoring Checklist</div>
  <div style="font-size: 15pt; font-weight: 400; line-height: 1.45; opacity: 0.95;">A recommended <b>6-step order</b> for building a Genie Agent. You <b>test, benchmark, and iterate</b> throughout, that loop wraps all six steps. This demo focuses on <b>steps 1 and 2</b>; steps 3 to 6 come in the next notebook.</div>
</div>

</div>
<br></br>
<div style="max-width: 1280px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">


<!-- 6-STEP FLOW (all steps full color, no dimming) -->
<div style="display: grid; grid-template-columns: 1fr auto 1fr auto 1fr auto 1fr auto 1fr auto 1fr; gap: 0; align-items: stretch;">

  <!-- STEP 1 -->
  <div style="background: #F9F7F4; border: 3px solid #2574B5; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #2574B5; line-height: 1; margin-bottom: 8px;">1</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Define purpose<br>and scope</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 2 -->
  <div style="background: #F9F7F4; border: 3px solid #02A36F; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #02A36F; line-height: 1; margin-bottom: 8px;">2</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add focused<br>tables</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 3 -->
  <div style="background: #F9F7F4; border: 3px solid #F8A805; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #F8A805; line-height: 1; margin-bottom: 8px;">3</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add metadata and<br>prompt matching</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 4 -->
  <div style="background: #F9F7F4; border: 3px solid #FF5F46; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #FF5F46; line-height: 1; margin-bottom: 8px;">4</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add SQL logic<br>(joins, expressions)</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 5 -->
  <div style="background: #F9F7F4; border: 3px solid #98182A; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #98182A; line-height: 1; margin-bottom: 8px;">5</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add example<br>SQL queries</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 6 -->
  <div style="background: #F9F7F4; border: 3px solid #1B3139; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #1B3139; line-height: 1; margin-bottom: 8px;">6</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add general<br>instructions</div>
  </div>

</div>

<!-- WRAPPING LOOP: test/iterate wraps all six steps -->
<div style="width: 100%; max-width: 1280px; margin: 14px auto 0 auto; background: rgba(27,49,57,0.06); border: 2px dashed #618794; border-radius: 10px; padding: 12px 18px; display: flex; align-items: center; justify-content: center; gap: 10px; box-sizing: border-box;">
  <span style="font-size: 22pt; color: #1B5162; line-height: 1;">&#8635;</span>
  <span style="font-size: 14pt; font-weight: 700; color: #1B5162;">Test, benchmark, and iterate. Repeat after every step.</span>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

### The 6-step authoring checklist

This is the recommended order for building any Genie Agent. Walk the room through all six at a high level, then set expectations for what this demo covers.

- **1. Define purpose and scope.** Pick a focused audience, topic, and a small set of business questions. Those questions become your sample questions and benchmarks.
- **2. Add focused tables.** Work backward from the questions to the smallest set of tables or views (aim for 5 or fewer). The foundation matters most.
- **3. Add metadata and prompt matching.** Descriptions, synonyms, and value matching. The strongest early accuracy lever and usually the easiest win.
- **4. Add SQL logic.** Joins plus reusable expressions (metrics, filters, dimensions). The preferred way to encode business definitions.
- **5. Add example SQL queries.** Pair a question with the SQL that answers it to teach Genie trusted patterns.
- **6. Add general instructions.** Global text rules, used only as a last resort for what SQL and metadata cannot express.

### What this demo covers

- **This notebook builds steps 1 and 2:** you define the audience and questions, then create the agent on a focused set of tables.
- **Steps 3 to 6** are the curation work in the next notebook.
- The dashed band underneath is the point to keep repeating: **test, benchmark, and iterate** after every step. The loop never really ends.

</details>



## E. Create a Genie Agent
<br></br>
<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <div style="margin-bottom: 10px;"><svg viewBox="0 0 806 200" fill="none" xmlns="http://www.w3.org/2000/svg" style="height:48px; width:auto; display:block;"><path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/>
<path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/>
<path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/>
<path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/>
<rect x="563.157" y="58.4648" width="242.843" height="84.9055" rx="42.4527" fill="#0B2026"/>
<path d="M664.499 114.947C661.815 114.947 659.493 114.366 657.532 113.205C655.596 112.018 654.1 110.38 653.042 108.289C652.01 106.173 651.493 103.735 651.493 100.974C651.493 98.187 652.01 95.7485 653.042 93.6584C654.1 91.5424 655.596 89.8909 657.532 88.7039C659.493 87.517 661.815 86.9235 664.499 86.9235C667.724 86.9235 670.356 87.7234 672.395 89.3232C674.433 90.8973 675.724 93.1294 676.266 96.0195H671.156C670.795 94.497 670.06 93.2971 668.95 92.4198C667.84 91.5424 666.344 91.1037 664.46 91.1037C662.757 91.1037 661.286 91.5037 660.048 92.3036C658.835 93.0778 657.893 94.2132 657.222 95.7098C656.577 97.1807 656.254 98.9353 656.254 100.974C656.254 103.012 656.577 104.767 657.222 106.238C657.893 107.683 658.835 108.805 660.048 109.605C661.286 110.38 662.757 110.767 664.46 110.767C666.344 110.767 667.84 110.367 668.95 109.567C670.06 108.741 670.795 107.606 671.156 106.161H676.266C675.749 108.896 674.459 111.05 672.395 112.625C670.356 114.173 667.724 114.947 664.499 114.947ZM697.803 114.947C695.196 114.947 692.887 114.366 690.874 113.205C688.887 112.018 687.326 110.38 686.191 108.289C685.055 106.173 684.488 103.722 684.488 100.935C684.488 98.1741 685.055 95.7485 686.191 93.6584C687.326 91.5424 688.887 89.8909 690.874 88.7039C692.887 87.517 695.196 86.9235 697.803 86.9235C700.46 86.9235 702.796 87.517 704.808 88.7039C706.821 89.8909 708.382 91.5424 709.492 93.6584C710.602 95.7485 711.156 98.1741 711.156 100.935C711.156 103.722 710.602 106.173 709.492 108.289C708.382 110.38 706.821 112.018 704.808 113.205C702.796 114.366 700.46 114.947 697.803 114.947ZM697.841 110.767C699.57 110.767 701.08 110.38 702.37 109.605C703.66 108.805 704.654 107.67 705.35 106.199C706.047 104.728 706.395 102.974 706.395 100.935C706.395 98.8966 706.047 97.1419 705.35 95.6711C704.654 94.2003 703.66 93.0778 702.37 92.3036C701.08 91.5037 699.57 91.1037 697.841 91.1037C696.112 91.1037 694.603 91.5037 693.313 92.3036C692.022 93.0778 691.016 94.2003 690.294 95.6711C689.597 97.1419 689.249 98.8966 689.249 100.935C689.249 102.974 689.597 104.728 690.294 106.199C691.016 107.67 692.022 108.805 693.313 109.605C694.603 110.38 696.112 110.767 697.841 110.767ZM720.339 114.482V87.3879H729.087C732.235 87.3879 734.829 87.9427 736.867 89.0523C738.931 90.1619 740.454 91.7359 741.434 93.7745C742.441 95.7872 742.944 98.187 742.944 100.974C742.944 103.709 742.441 106.096 741.434 108.135C740.454 110.147 738.931 111.708 736.867 112.818C734.829 113.928 732.235 114.482 729.087 114.482H720.339ZM724.984 110.534H728.855C731.203 110.534 733.048 110.16 734.39 109.412C735.757 108.638 736.725 107.541 737.293 106.122C737.886 104.677 738.183 102.961 738.183 100.974C738.183 98.9611 737.886 97.2452 737.293 95.8259C736.725 94.3809 735.757 93.2713 734.39 92.4972C733.048 91.6972 731.203 91.2973 728.855 91.2973H724.984V110.534ZM752.124 114.482V87.3879H769.619V91.1425H756.769V98.9224H768.458V102.6H756.769V110.728H769.619V114.482H752.124Z" fill="white"/>
<path d="M636.472 101.239L620.99 116.72L613.554 109.283L621.598 101.239L613.554 93.1943L620.99 85.7578L636.472 101.239Z" fill="#FF3621"/>
<path d="M584.879 101.241L600.36 85.7598L607.797 93.1968L599.753 101.241L607.797 109.286L600.36 116.723L584.879 101.241Z" fill="#FF3621"/></svg></div>
  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 14px;">Build With Genie Code!</div>
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">You can use <strong>Genie Code</strong> to build your <strong>Genie Agent</strong> for you, but in this workshop we go through the steps in the <strong>UI</strong> by hand. This teaches the fundamentals of what makes a Genie Agent accurate and trustworthy. You will:
  <ul style="margin: 10px 0 14px 0; padding-left: 22px;">
    <li style="margin-bottom: 8px;">Create the agent</li>
    <li style="margin-bottom: 8px;">Add tables</li>
    <li style="margin-bottom: 8px;">Curate the <strong>Knowledge Store</strong> with metadata, SQL logic, and instructions</li>
  </ul>
  Once you understand those steps, you can hand the work to <strong>Genie Code</strong> or other AI tools to scaffold an agent for you. Knowing how the pieces fit together lets you:
  <ul style="margin: 10px 0 0 0; padding-left: 22px;">
    <li style="margin-bottom: 8px;">Give better prompts and context</li>
    <li style="margin-bottom: 0;">Review and correct what the AI produces</li>
  </ul>
</div>

  <a href="https://www.databricks.com/blog/introducing-genie-code" target="_blank" style="display: inline-block; background: #FF5F46; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">Introducing Genie Code &rarr;</a>
  <div style="display: flex; gap: 10px; margin-top: 12px;">
    <a href="https://docs.databricks.com/aws/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">AWS &rarr;</a>
    <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">Azure &rarr;</a>
    <a href="https://docs.databricks.com/gcp/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">GCP &rarr;</a>
  </div>
</div>
</div>


In this section, you will create a new Genie Agent through the Databricks UI and add the **bakehouse** tables to it. You are building **steps 1 and 2** from the checklist above: a focused purpose and a focused set of tables.

### Audience and Questions (Review)

**Audience:** Bakehouse franchise managers

- How many customers do we have?
- What are our top 5 franchise locations by revenue?
- Who are our best customers?
- Count of bad reviews by location?
- How many franchises does each supplier serve?
- What are total sales by region?

### E1. Create a New Genie Agent

Complete the following steps to create a Genie Agent:

1. In the main navigation sidebar, right-click **Genie Agents** and select **Open Link in New Tab**.
   - If you have existing Genie Agents, they will be listed here. For this demo, you will create a new one.

2. Select **+ New** in the top-right corner of the Genie home page.

3. Add the following five tables from your **labuser_YOUR_USER_NAME** catalog and **genie_course_bakehouse** schema, then select **Create**:

   - **sales_customers_gold**
   - **sales_franchises_gold**
   - **sales_suppliers_gold**
   - **sales_transactions_gold**
   - **media_customer_reviews_gold**

4. In the **Created Genie Agent** panel on the right (you might have to select **Configure**) complete the following:
   - **Accept** the **Genie Suggested Description**.
   - Under **About this agent**, select the **Edit Pencil icon**.
   - Configure the following, then select **Save**:

   | Field | Value |
   |-------|-------|
   | Name | `YOUR_NAME - Bakehouse Sales` |
   | Default warehouse | Select your **Serverless SQL Warehouse** |

5. Select the **Sources** tab and confirm your setup contains the 5 tables.

## F. Explore the Genie UI

After creating the Genie Agent, you are taken to the Genie Agent interface.

### F1. The Genie Agent Chat Interface

The main area of a Genie Agent is the **chat interface**. This is where users type natural language questions and receive answers.

1. Review the chat interface options:

<br></br>

<div class="two-pane">

<style>

.two-sides {
    display: flex;
    justify-content: center;
    align-items: stretch;
    gap: 40px;
    flex-wrap: wrap;
}

.box {
    width: 480px;
    min-height: 320px;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
    gap: 6px;
    padding: 22px;
    text-align: left;
    position: relative;
    box-sizing: border-box;
}

.box::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
}

.box.chat::before { background: #FF5F46; }
.box.agent::before { background: #FF5F46; }

.box-icon {
    text-align: center;
    margin-top: 4px;
    margin-bottom: 0;
}

.box-icon svg {
    height: 160px;
    width: auto;
    display: block;
    margin: 0 auto;
}

.box-title {
    font-size: 20pt;
    font-weight: 700;
    color: #0b2026;
    text-align: center;
    margin-top: 0;
}

.box-text {
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.55;
}

.box-text ul {
    margin: 0;
    padding-left: 18px;
}

.box-text li {
    margin-bottom: 12px;
}

.box-text li:last-child {
    margin-bottom: 0;
}

.note {
    margin-top: 10px;
    padding: 10px 12px;
    background: #EEEDE9;
    border-left: 4px solid #FF5F46;
    border-radius: 6px;
    font-size: 13pt;
}

</style>

<div class="two-sides">

<div class="box chat">
<div class="box-icon"><svg viewBox="0 0 150 150" role="img" aria-label="Chat" fill="none" style="height:160px; width:auto;">
<path d="M111.54 36.6797H120.73C127.19 36.6797 132.43 41.9197 132.43 48.3797V105.16C132.43 111.62 127.19 116.86 120.73 116.86H110.84V133.35L90.4199 116.86H67.1899" stroke="#FF5F46" stroke-width="3" stroke-miterlimit="3"/>
<path d="M36.3199 36.6799H111.26V28.3699C111.26 21.9099 106.02 16.6699 99.5599 16.6699H29.2699C22.8099 16.6699 17.5699 21.9099 17.5699 28.3699V58.2099" stroke="#FF5F46" stroke-width="3" stroke-miterlimit="3"/>
<path d="M36.3199 56.7397H92.5099" stroke="#FF5F46" stroke-width="3" stroke-miterlimit="3"/>
<path d="M92.5099 76.7698H17.5699V85.1098C17.5699 91.5698 22.8099 96.8098 29.2699 96.8098L42.0899 96.7298V113.3L62.5099 96.8098H95.6199C104.26 96.8098 111.27 95.9598 111.27 87.3198V55.2798" stroke="#FF5F46" stroke-width="3" stroke-miterlimit="3"/>
</svg></div>
<div class="box-title">Chat</div>
<div class="box-text">
<ul>
<li>Use standard chat for quick, focused questions and fast iteration</li>
<li>Use a Genie Agent to explore business data:
<a href="https://docs.databricks.com/aws/en/genie/talk-to-genie">AWS</a> |
<a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/talk-to-genie">Azure</a> |
<a href="https://docs.databricks.com/gcp/en/genie/talk-to-genie">GCP</a></li>

</ul>
</div>
</div>

<div class="box agent">
<div class="box-icon"><svg viewBox="0 0 512 512" role="img" aria-label="Agent" style="height:150px; width:auto;"><rect x="64" y="352" width="192" height="96" fill="#FF5F46"/>
<rect x="64" y="160" width="192" height="96" fill="#FF5F46"/>
<rect x="256" y="256" width="192" height="96" fill="#FABFBA"/>
<rect x="256" y="64" width="192" height="96" fill="#FABFBA"/></svg></div>
<div class="box-title">Agent</div>
<div class="box-text">
<ul>
<li>Use Agent mode for deeper investigation, more complex analysis and hypothesis testing</li>
<li>Agent mode in Genie Agents:
<a href="https://docs.databricks.com/aws/en/genie/agent-mode">AWS</a> |
<a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/concepts#agent-mode">Azure</a> |
<a href="https://docs.databricks.com/gcp/en/genie/agent-mode">GCP</a></li>
</ul>

</div>
</div>

</div>

</div>


### F2. Explore the Configuration Panel

The configuration panel allows you to configure how the Genie Agent behaves.

1. Review the top options in a Genie Agent:

   a. **Configure**
   - Set up the space.
   - Choose the SQL warehouse.
   - Add or remove tables.
   - Define instructions and example SQL.

   b. **Benchmark**
   - Benchmarks allow you to create a set of test questions that you can run to assess Genie's overall response accuracy.
   - Add key questions with expected answers.
   - Run benchmark suites and track accuracy over time.
   - Use benchmarks in a Genie Agent:
   [AWS](https://docs.databricks.com/aws/en/genie/benchmarks) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/monitor) |
   [GCP](https://docs.databricks.com/gcp/en/genie/benchmarks)

   c. **Monitor**
   - See how the space is used.
   - View all questions, ratings, feedback and problem cases.
   - Use it to decide what to fix or improve.

2. Select the **![Gear](https://files.training.databricks.com/binder/prod_main/enable-trusted-insights-for-your-organization-with-genie-en_us-1.0.0/images/20261001T150649Z/Enable Trusted Insights for your Organization with Genie/Includes/images/icons/gear.png) Configure** icon to open the settings panel (if not already open). Review the **Knowledge Store** (Genie context) tabs in the right panel:

   a. **About**
   - Shows the owner, warehouse, and description.
   - Lists common or starter questions.

   b. **Sources**
   - The list of Unity Catalog tables available to the Genie Agent. You should see the five tables you added: the four `sales_*` tables and `media_customer_reviews_gold`.
   - Lets you inspect the schema Genie uses.

   c. **Instructions**
   - Add general instructions on how you want this Genie Agent to behave.

   d. **Examples**
   - Enables you to teach your agent how to answer questions with curated examples.
   - Shows how Genie has been "taught."
   - Example SQL **Joins**, **Expressions**, **Filters**, and **Queries** that define your core business logic.

### F3. Viewing Table Details

You can inspect the tables and columns that Genie has access to directly from the Genie Agent.

1. In the settings panel, under **Sources**, select one of the tables.

2. Review the table details:

   - **Column names and data types** - Each column is listed with its name and type.
   - **Column descriptions** - Notice that the column descriptions are **currently empty** (no metadata was set on the UC tables).
      - Genie Agents rely on **column details** to interpret questions.

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Important: No Metadata in Unity Catalog, Instructions, or Benchmarks Added Intentionally</strong>
  <div style="color:#333;">

At this stage, we are **not** adding any of the following to the Genie Agent:

- **Table or column metadata** - descriptions and synonyms on the tables that help Genie understand what each table and column represents

- **SQL logic** - joins, SQL expressions, or example queries that encode business metrics and patterns

- **Sample questions** - example queries that help Genie understand common question patterns

- **Text instructions** - behavioral guidance like formatting rules or clarification prompts

- **Benchmarks** - tests to monitor responses of the Genie Agent

This is intentional. By starting without any of these, you will see how Genie interprets questions using only table and column names. Later in this workshop, you will iteratively add each component and observe how you can improve Genie's accuracy and relevance, and provide more consistent results.
  </div>
</div>

## G. Ask Your Genie Agent a Question

Now that your Genie Agent is created, it is time to interact with it and see how it works!

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">How Genie Answers Questions</strong>
<p style="margin: 8px 0 0 0; color: #333;">When you ask Genie a question, it follows this process:</p>
<ol style="margin: 8px 0 0 16px; color: #333;">
<li><strong>Interprets</strong> your natural language question</li>
<li><strong>Generates SQL</strong> based on the available table schemas, instructions, and sample queries</li>
<li><strong>Executes the SQL</strong> against the configured SQL warehouse</li>
<li><strong>Returns the results</strong> as a table, along with the generated SQL for transparency</li>
</ol>
<p style="margin: 8px 0 0 0; color: #333;">You can always review the SQL that Genie generated by expanding the query details in the response.</p>
</div>
</div>
</div>

### G1. Baseline: How Many Customers Do We Have?

This first question establishes the basic workflow: ask a question, review the result, and inspect the generated SQL.

1. In the Genie Agent chat interface select **Chat** 
    - Then ask the following question: <span style="background-color:#d4edda; padding:4px 6px; border-radius:4px;"><b>How many customers do we have?</b></span>

| # | Question | Why We Are Asking This | Potential Issues Without Metadata |
|---|----------|-----------------------|----------------------------------|
| 1 | *How many customers do we have?* | **Baseline** - Simple, unambiguous question to ask → SQL → result workflow. | Minimal - Genie should answer this correctly out of the box. |

2. After Genie responds:
   - Confirm the result is **300**.

   - Review the **analysis**.
      - Select **⌄** > **Show code** to view the SQL.
      - Verify Genie queried **sales_customers_gold** with a query like this. Results can differ:

```sql
SELECT COUNT(DISTINCT `customerID`) AS total_customers
FROM `labuser_YOUR_USER_NAME`.`genie_course_bakehouse`.`sales_customers_gold`;
```


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Genie Response
  </strong>
  <div style="color:#333;">

This is a straightforward question that Genie should handle easily.

The table name **sales_customers_gold** and the question are clearly aligned.

  </div>
</div>

### G2. After a Genie Response

1. Look at the Genie results (after viewing the SQL code from above).

<img src="https://files.training.databricks.com/binder/prod_main/enable-trusted-insights-for-your-organization-with-genie-en_us-1.0.0/images/20261001T150649Z/Enable Trusted Insights for your Organization with Genie/Includes/images/demo-creating-genie-agent/question-genie-checkpoint.png" width="1300">

2. Select the **More Options** button _(Three dots next to "Hide Code")_. 
    - Notice you can perform different operations with this response:
    - **NOTE:** **These options are for the Genie curator. General users will not see the options below.**

<br></br>

<div class="two-pane">

<style>

.two-sides {
    display: flex;
    justify-content: center;
    align-items: stretch;
    gap: 40px;
    flex-wrap: wrap;
}

.box {
    width: 480px;
    min-height: 300px;
    background: #F9F7F4;
    border: none;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    overflow: hidden;
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
    gap: 6px;
    padding: 22px;
    text-align: left;
    position: relative;
    box-sizing: border-box;
}

.box::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
}

.box.instruction::before {
    background: #FF5F46;
}

.box.benchmark::before {
    background: #1B5162;
}

.box-icon {
    text-align: center;
    margin-top: 4px;
    margin-bottom: 0;
}

.box-icon img {
    height: 175px;
    width: auto;
    object-fit: contain;
}

.box-title {
    font-size: 20pt;
    font-weight: 700;
    color: #0b2026;
    margin-top: 0;
    margin-bottom: 4px;
    text-align: center;
}

.box-text {
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.55;
}

.box-text ul {
    margin: 0;
    padding-left: 18px;
}

.box-text li {
    margin-bottom: 12px;
}

.box-text li:last-child {
    margin-bottom: 0;
}

</style>

<div class="two-sides">

<div class="box instruction">
<div class="box-icon"><img src="https://files.training.databricks.com/binder/prod_main/enable-trusted-insights-for-your-organization-with-genie-en_us-1.0.0/images/20261001T150649Z/Enable Trusted Insights for your Organization with Genie/Includes/images/icons/instructions.png"></div>
<div class="box-title">Add Instruction</div>
<div class="box-text">
<ul>
<li>Saves this question and Genie's SQL as an instruction or example for the space</li>
<li>Helps Genie reuse this pattern for similar questions in the future</li>
<li>Changes <strong>future behavior</strong> by teaching Genie, not the current answer</li>
</ul>
</div>
</div>

<div class="box benchmark">
<div class="box-icon"><img src="https://files.training.databricks.com/binder/prod_main/enable-trusted-insights-for-your-organization-with-genie-en_us-1.0.0/images/20261001T150649Z/Enable Trusted Insights for your Organization with Genie/Includes/images/icons/benchmarks.png"></div>
<div class="box-title">Add to Benchmark</div>
<div class="box-text">
<ul>
<li>Saves this question, optionally with the correct SQL, as a benchmark test case</li>
<li>Used later to run benchmark suites and measure accuracy over time</li>
<li>For <strong>evaluation only</strong>. <strong>Benchmarks do not</strong> change how Genie answers</li>
</ul>
</div>
</div>

</div>

</div>

3. Look at the **Is this useful?** option. This enables general users to rate the response. 

4. In **Is this useful?** select 👍.

    You can then review the feedback in **Monitoring**.
    - **NOTE:** We will look at these features in more detail later.
<br></br>
<br></br>
<div class="three-pane">
<style>
.three-sides {
    display: flex;
    justify-content: center;
    align-items: stretch;
    gap: 40px;
    flex-wrap: wrap;
}
.box {
    width: 340px;
    min-height: 300px;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
    gap: 14px;
    padding: 22px;
    text-align: left;
    position: relative;
    box-sizing: border-box;
}
.box::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
}
.box.yes::before { background: #00A972; }
.box.no::before { background: #98102A; }
.box.review::before { background: #FFAB00; }
.box-title {
    font-size: 20pt;
    font-weight: 700;
    color: #0b2026;
    text-align: center;
    margin-top: 6px;
}
.box-text {
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.55;
}
.box-text ul {
    margin: 0;
    padding-left: 18px;
}
.box-text li {
    margin-bottom: 12px;
}
.box-text li:last-child {
    margin-bottom: 0;
}
</style>
<div class="three-sides">
<div class="box yes">
<div class="box-title">👍 Thumbs Up</div>
<div class="box-text">
<ul>
<li>Upvotes the answer</li>
<li>Tells Genie authors this response was helpful</li>
<li>Helps identify good examples to turn into instructions or benchmarks later</li>
</ul>
</div>
</div>
<div class="box no">
<div class="box-title">👎 Thumbs Down</div>
<div class="box-text">
<ul>
<li>Downvotes the answer</li>
<li>Prompts the user to select a reason <strong>Why</strong> and add a comment</li>
<li>Flags the response in the Monitoring tab for author review</li>
<li>Signals that the space needs improvement for this type of question</li>
</ul>
</div>
</div>
<div class="box review">
<div class="box-title">🔍 Request Review</div>
<div class="box-text">
<ul>
<li>Escalates the response to space owners or editors for manual review</li>
<li>Available in <strong>Chat mode</strong> only</li>
</ul>
</div>
</div>
</div>
</div>

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    What Just Happened?
  </strong>
  <div style="color:#333;">

You just completed the basic Genie workflow:

1. **Asked** a natural language question
2. **Reviewed** the generated SQL and result
3. **Rated** the response (thumbs up, consumers can provide this feedback)

This is the same loop you will use throughout the course to evaluate and improve your Genie Agent.

  </div>
</div>

## H. Conclusion

In this demo, you:

- **Explored** the five bakehouse gold tables, their relationships, and a data quality issue (**21 of 48 franchises** reference a `supplierID` with no matching supplier).
- **Created** a Genie Agent on those tables.
- **Explored** the Genie UI: the chat interface, configuration panel, and sources.
- **Asked** a baseline question, reviewed the generated SQL, and gave feedback with thumbs up/down and request review.

You now have a working Genie Agent. In the next notebook, you will benchmark it to establish a baseline, see where an un-curated agent falls short, and curate it to improve accuracy.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>